In [1]:
from system_check import *
CUDA_state_print_limited()


CUDA доступна: True
Количество GPU: 4


In [4]:
import pandas as pd

In [2]:
from collections import Counter
from typing import Union, List
from rouge_score import rouge_scorer
from bert_score import score
import evaluate


# === 1. Character-level F1  ===
def char_f1(pred: str, gold: Union[str, List[str]]) -> float:
    if pd.isna(pred) or not str(pred).strip():
        return 0.0
    if isinstance(gold, str):
        gold = [gold]
    if not gold or all(pd.isna(g) or not str(g).strip() for g in gold):
        return 0.0
    
    scores = []
    pred_lower = str(pred).lower()
    pred_chars = Counter(pred_lower)
    
    for g in gold:
        if pd.isna(g) or not str(g).strip():
            continue
        g_lower = str(g).lower()
        gold_chars = Counter(g_lower)
        common = sum((pred_chars & gold_chars).values())
        if common == 0:
            continue
        precision = common / len(pred_lower)
        recall = common / len(g_lower)
        f1 = 2 * precision * recall / (precision + recall)
        scores.append(f1)
    
    return max(scores) if scores else 0.0

# === 2. Token-level F1 (SQuAD) ===
_squad_metric = None

def token_f1(pred: str, gold: Union[str, List[str]]) -> float:
    global _squad_metric
    if _squad_metric is None:
        _squad_metric = evaluate.load("squad")
    
    if isinstance(gold, str):
        gold = [gold]
    
    # Обработка пустых значений
    pred = str(pred).strip() if pd.notna(pred) else ""
    gold = [str(g).strip() for g in gold if pd.notna(g) and str(g).strip()]
    if not pred or not gold:
        return 0.0
    
    # Вызов squad.compute для одной строки
    result = _squad_metric.compute(
        predictions=[{"id": "0", "prediction_text": pred}],
        references=[{
            "id": "0",
            "answers": {
                "text": gold,
                "answer_start": [0] * len(gold)
            }
        }]
    )
    return result["f1"]

# === 3. Metrics BERTScore  ===
def bertscore_metrics(pred: str, gold: Union[str, List[str]]) -> dict:
    if isinstance(gold, str):
        gold = [gold]
    
    pred = str(pred).strip() if pd.notna(pred) else ""
    gold = [str(g).strip() for g in gold if pd.notna(g) and str(g).strip()]
    
    if not pred or not gold:
        return {"p": 0.0, "r": 0.0, "f1": 0.0}
    
    # Считаем метрики для каждого эталона
    best_p = best_r = best_f1 = 0.0
    for ref in gold:
        P, R, F1 = score(
            cands=[pred],
            refs=[ref],
            lang="ru",
            rescale_with_baseline=False,
            verbose=False
        )
        best_p = max(best_p, P.item())
        best_r = max(best_r, R.item())
        best_f1 = max(best_f1, F1.item())
    
    return {"p": best_p, "r": best_r, "f1": best_f1}

# === 4. ROUGE-L F1 ===
_rouge_metric = None

def rouge_l_f1(pred: str, gold: Union[str, List[str]]) -> float:
    """
    Вычисляет ROUGE-L F1 между предсказанием и эталоном(ами).
    Для списка эталонов возвращает максимальный результат.
    
    Args:
        pred: Предсказанный ответ
        gold: Эталонный ответ (строка) или список эталонов
    
    Returns:
        ROUGE-L F1 в диапазоне [0.0, 1.0]
    """
    _rouge_metric = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)

    # Преобразуем эталон в список
    if isinstance(gold, str):
        gold = [gold]
    
    # Обработка пустых/некорректных значений
    pred = str(pred).strip() if pd.notna(pred) else ""
    gold = [str(g).strip() for g in gold if pd.notna(g) and str(g).strip()]
    
    if not pred or not gold:
        return 0.0
    
    # Вычисляем ROUGE-L для каждого эталона, берём максимум
    best_f1 = 0.0
    for ref in gold:
        try:
            scores = _rouge_metric.score(ref, pred)
            best_f1 = max(best_f1, scores['rougeL'].fmeasure)
        except Exception:
            continue  # пропускаем проблемные пары
    
    return best_f1

/home/pmartynyuk/UnTIE project/untie_venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
df_res_rubert = pd.read_csv('/home/pmartynyuk/UnTIE project/scripts/df_res_rubert.csv',
                            encoding='utf-8', sep=',')
df_res_rubert.head()

,text,Task_aspects,Task_count,text_clean,rubert_answer
0,В рамках проекта по <Task> автоматизации раб...,['автоматизации работы с поэтическими текстами...,2,В рамках проекта по автоматизации работы с ...,"призвана решать информационная система ,"
1,Проблема <Task> своевременного предупреждения...,['своевременного предупреждения населения об о...,2,Проблема своевременного предупреждения насел...,своевременного предупреждения населения об опа...
2,Работа посвящена <Contrib> описанию структур...,['генотипирования вируса клещевого энцефалита ...,2,Работа посвящена описанию структуры информа...,генотипирования вируса клещевого энцефалита (В...
3,Для эффективного <Task> освоения залежей угл...,['освоения залежей углеводородов'],1,Для эффективного освоения залежей углеводор...,геонавигация скважины со сложной траекторией в...
4,При <Task> формировании единого информационно...,['формировании единого информационного простра...,1,При формировании единого информационного про...,с использованием метода матрицы структуры прое...


In [6]:
from tqdm import tqdm
tqdm.pandas() 

In [7]:
def compute_metrics_rowwise(
    df: pd.DataFrame,
    pred_col: str = "rubert_answer",
    gold_col: str = "Task_aspects"
) -> pd.DataFrame:
    """
    Вычисляет метрики качества для каждой строки датафрейма.
    
    Параметры:
    - df: исходный DataFrame
    - pred_col: имя колонки с предсказанным ответом (str)
    - gold_col: имя колонки с эталонными ответами (List[str] или str)
    
    Возвращает:
    - DataFrame с добавленными колонками метрик
    """
    df = df.copy()
    
    # Character F1
    df["char_f1"] = df.progress_apply(
        lambda row: char_f1(row[pred_col], row[gold_col]),
        axis=1
    )
    
    # Token F1 (SQuAD)
    df["token_f1"] = df.progress_apply(
        lambda row: token_f1(row[pred_col], row[gold_col]),
        axis=1
    )

    df['rouge_l_f1'] = df.progress_apply(
    lambda row: rouge_l_f1(row[pred_col], row[gold_col]),
    axis=1
    )
    
    # BERTScore (P, R, F1)
    bert_results = df.progress_apply(
        lambda row: bertscore_metrics(row[pred_col], row[gold_col]),
        axis=1
    )
    df["bertscore_p"] = bert_results.apply(lambda x: x["p"])
    df["bertscore_r"] = bert_results.apply(lambda x: x["r"])
    df["bertscore_f1"] = bert_results.apply(lambda x: x["f1"])
    
    # Округление
    for col in ["char_f1", "token_f1", "bertscore_p", "bertscore_r", "bertscore_f1"]:
        df[col] = df[col].round(4)
    
    return df

In [8]:
# === Запуск ===
df_result = compute_metrics_rowwise(df_res_rubert)

  0%|          | 0/168 [00:00<?, ?it/s]